In [1]:
import pandas as pd

# Setup & Data Loading

In [2]:
df = pd.read_csv("dataset_sig_st.csv")

In [3]:
# Fix: G2_anxious_without_device has zero respondents in "Always" x "1-3 hours",
# causing perfect separation (OR blows up to ~1e5 with an astronomically wide CI).
# Merge the sparse "Always" (n=9) into the adjacent "Often" category before modeling.
df['G2_anxious_without_device'] = df['G2_anxious_without_device'].replace(
    {'Always': 'Often'}
)
print(df['G2_anxious_without_device'].value_counts())


G2_anxious_without_device
Not at all      181
Occasionally    147
Sometimes        80
Often            44
Name: count, dtype: int64


In [4]:
df.columns

Index(['D1_daily_screen', 'age_class', 'A2_gender', 'B1_residence',
       'income_class', 'weight_class', 'D2_weekend_screen',
       'D5_parents_screen_time', 'D6_ai_daily', 'E5_bedtime', 'E6_wake_time',
       'F1_academic_satisfaction', 'F4_interest_reduced', 'G1_mood_swings',
       'G2_anxious_without_device', 'H1_mobile_while_eating',
       'H2_appetite_change', 'BMI_class', 'D4_content_grouped'],
      dtype='str')

In [5]:

# Check target variable distribution
print("Target variable: D1_daily_screen")
print(df['D1_daily_screen'].value_counts())
print("\nShape:", df.shape)
print("\nMissing values:\n", df.isnull().sum().sum())


Target variable: D1_daily_screen
D1_daily_screen
1-3 hours            137
3-5 hours            125
Less than 1 hour     112
More than 5 hours     78
Name: count, dtype: int64

Shape: (452, 19)

Missing values:
 0


# Multinomial Logistic Regression
**Target variable:** `D1_daily_screen`

---

In [6]:
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (classification_report, confusion_matrix,
                             accuracy_score, ConfusionMatrixDisplay)
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

In [7]:
#  1. Prepare features and target
target = 'D1_daily_screen'
features = [c for c in df.columns if c != target]

X = df[features].copy()
y = df[target].copy()

# Encode every categorical column with a fresh LabelEncoder
encoders = {}
for col in X.columns:
    le = LabelEncoder()
    X[col] = le.fit_transform(X[col].astype(str))
    encoders[col] = le

le_target = LabelEncoder()
y_enc = le_target.fit_transform(y.astype(str))
class_names = le_target.classes_

print("Classes:", class_names)
print("Encoded labels:", np.unique(y_enc))


Classes: ['1-3 hours' '3-5 hours' 'Less than 1 hour' 'More than 5 hours']
Encoded labels: [0 1 2 3]


In [8]:

# 2. Train / Test Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y_enc, test_size=0.2, random_state=42, stratify=y_enc
)
print(f"Train size: {X_train.shape[0]}  |  Test size: {X_test.shape[0]}")


Train size: 361  |  Test size: 91


In [9]:

# 3. Fit Multinomial Logistic Regression
mlr = LogisticRegression(
    solver='lbfgs',
    max_iter=1000,
    random_state=42
)
mlr.fit(X_train, y_train)
y_pred = mlr.predict(X_test)

# 4. Evaluation 
acc = accuracy_score(y_test, y_pred)
print(f"Test Accuracy: {acc:.4f} ({acc*100:.2f}%)\n")
print("Classification Report:")
print(classification_report(y_test, y_pred, target_names=class_names))


Test Accuracy: 0.4945 (49.45%)

Classification Report:
                   precision    recall  f1-score   support

        1-3 hours       0.47      0.57      0.52        28
        3-5 hours       0.45      0.56      0.50        25
 Less than 1 hour       0.59      0.45      0.51        22
More than 5 hours       0.56      0.31      0.40        16

         accuracy                           0.49        91
        macro avg       0.52      0.47      0.48        91
     weighted avg       0.51      0.49      0.49        91



## Odds Ratios
**Per class vs. reference class (`1-3 hours`)**

In [10]:

# Odds Ratios 
# OR = exp(coefficient); OR > 1 → increases odds of that class, < 1 → decreases
odds_ratio_df = pd.DataFrame(
    np.exp(mlr.coef_),
    index=class_names,
    columns=features
).T

print("Odds Ratios per class (rows=features, columns=D1_daily_screen classes):\n")
print(odds_ratio_df.round(4).to_string())


Odds Ratios per class (rows=features, columns=D1_daily_screen classes):

                           1-3 hours  3-5 hours  Less than 1 hour  More than 5 hours
age_class                     0.5824     1.2293            0.6504             2.1474
A2_gender                     0.8959     1.9764            0.5139             1.0990
B1_residence                  0.5598     1.1569            0.9281             1.6638
income_class                  0.7514     0.8752            1.8045             0.8427
weight_class                  1.1215     0.6901            1.2230             1.0565
D2_weekend_screen             1.5432     0.1579           22.4901             0.1825
D5_parents_screen_time        1.1425     0.9341            1.9721             0.4751
D6_ai_daily                   0.8686     1.1866            1.0400             0.9329
E5_bedtime                    1.0364     0.8934            1.6285             0.6632
E6_wake_time                  0.9705     1.2144            1.0147            

In [11]:
import statsmodels.api as sm
from statsmodels.discrete.discrete_model import MNLogit

# Refit using statsmodels to get standard errors, p-values, and CIs
X_sm = sm.add_constant(X.values.astype(float))
feature_names_sm = ['const'] + features

mnl = MNLogit(y_enc, X_sm)
result = mnl.fit(method='bfgs', maxiter=2000, disp=False)

ref_class = class_names[0]
eq_classes = class_names[1:]

rows = []
for eq_idx, cls in enumerate(eq_classes):
    coefs = result.params[:, eq_idx]
    se    = result.bse[:, eq_idx]
    pvals = result.pvalues[:, eq_idx]

    for feat_idx, feat in enumerate(feature_names_sm):
        if feat == 'const':
            continue
        b = coefs[feat_idx]
        s = se[feat_idx]
        rows.append({
            'Class (vs. reference)': f"{cls}  vs.  {ref_class}",
            'Feature':               feat,
            'Odds Ratio':            round(np.exp(b), 4),
            'Lower 95% CI':          round(np.exp(b - 1.96 * s), 4),
            'Upper 95% CI':          round(np.exp(b + 1.96 * s), 4),
            'p-value':               round(pvals[feat_idx], 4),
        })

or_table = pd.DataFrame(rows)

pd.set_option('display.max_rows', 200)
pd.set_option('display.float_format', '{:.4f}'.format)
print("Odds Ratio Table — Multinomial Logistic Regression")
print(f"Reference class: '{ref_class}'\n")
print(or_table.to_string(index=False))


Odds Ratio Table — Multinomial Logistic Regression
Reference class: '1-3 hours'

            Class (vs. reference)                   Feature  Odds Ratio  Lower 95% CI  Upper 95% CI  p-value
        3-5 hours  vs.  1-3 hours                 age_class      2.8110        1.6268        4.8569   0.0002
        3-5 hours  vs.  1-3 hours                 A2_gender      2.7993        1.2559        6.2394   0.0118
        3-5 hours  vs.  1-3 hours              B1_residence      3.7819        1.5504        9.2249   0.0035
        3-5 hours  vs.  1-3 hours              income_class      1.0458        0.6817        1.6043   0.8375
        3-5 hours  vs.  1-3 hours              weight_class      0.9223        0.5488        1.5501   0.7602
        3-5 hours  vs.  1-3 hours         D2_weekend_screen      0.0492        0.0169        0.1431   0.0000
        3-5 hours  vs.  1-3 hours    D5_parents_screen_time      0.8308        0.3278        2.1057   0.6961
        3-5 hours  vs.  1-3 hours              

In [12]:
from matplotlib.backends.backend_pdf import PdfPages
import matplotlib.pyplot as plt

output_pdf = "report_v3/odds_ratio_table_st.pdf"

col_widths = [0.30, 0.22, 0.12, 0.13, 0.13, 0.10]
row_height = 0.32   # inches per row
header_height = 0.55
rows_per_page = 40

pages = [or_table.iloc[i:i+rows_per_page] for i in range(0, len(or_table), rows_per_page)]

with PdfPages(output_pdf) as pdf:
    for page_num, page_df in enumerate(pages):
        n_rows = len(page_df)
        fig_h = header_height + n_rows * row_height + 1.2
        fig, ax = plt.subplots(figsize=(14, fig_h))
        ax.axis('off')

        table_data = [or_table.columns.tolist()] + page_df.values.tolist()
        col_labels = or_table.columns.tolist()

        tbl = ax.table(
            cellText=page_df.values.tolist(),
            colLabels=col_labels,
            colWidths=col_widths,
            loc='center',
            cellLoc='center',
        )
        tbl.auto_set_font_size(False)
        tbl.set_fontsize(7.5)
        tbl.scale(1, 1.4)

        # Style header
        for col_idx in range(len(col_labels)):
            cell = tbl[0, col_idx]
            cell.set_facecolor('#2c3e50')
            cell.set_text_props(color='white', fontweight='bold')

        # Alternating row shading
        for row_idx in range(1, n_rows + 1):
            for col_idx in range(len(col_labels)):
                cell = tbl[row_idx, col_idx]
                cell.set_facecolor('#f2f2f2' if row_idx % 2 == 0 else 'white')
                cell.set_edgecolor('#cccccc')

        total_pages = len(pages)
        ax.set_title(
            f"Odds Ratio Table — Multinomial Logistic Regression\n"
            f"Reference class: '{ref_class}'   |   Page {page_num+1} of {total_pages}",
            fontsize=10, fontweight='bold', pad=12
        )
        plt.tight_layout()
        pdf.savefig(fig, bbox_inches='tight')
        plt.close(fig)

print(f"Saved: {output_pdf}  ({len(pages)} page(s), {len(or_table)} rows)")


Saved: report_v3/odds_ratio_table_st.pdf  (2 page(s), 54 rows)


## Goodness of Fit
**Multinomial Logistic Regression — model diagnostics**

In [13]:
# Likelihood Ratio Test & Pseudo R²

llr      = result.llr
llr_pval = result.llr_pvalue
mcfadden = result.prsquared          # McFadden's pseudo-R²

print("=" * 55)
print("  Goodness of Fit — Multinomial Logistic Regression")
print("=" * 55)
print(f"\n1. Likelihood Ratio Test")
print(f"   LR statistic : {llr:.4f}")
print(f"   p-value      : {llr_pval:.4e}")
print(f"   Interpretation: {'Model fits significantly better than null (intercept-only)' if llr_pval < 0.05 else 'No significant improvement over null model'}")

print(f"\n2. Pseudo R² (McFadden)")
print(f"   McFadden R²  : {mcfadden:.4f}")
if mcfadden < 0.1:
    interpretation = "Poor fit"
elif mcfadden < 0.2:
    interpretation = "Acceptable fit"
elif mcfadden < 0.4:
    interpretation = "Good fit  (0.2–0.4 is typical for MLR)"
else:
    interpretation = "Very good fit"
print(f"   Interpretation: {interpretation}")
print("=" * 55)

  Goodness of Fit — Multinomial Logistic Regression

1. Likelihood Ratio Test
   LR statistic : 498.6877
   p-value      : 2.9518e-73
   Interpretation: Model fits significantly better than null (intercept-only)

2. Pseudo R² (McFadden)
   McFadden R²  : 0.4038
   Interpretation: Very good fit


In [14]:

# Goodness of Fit — Observed vs. Expected Counts
from scipy.stats import chi2

y_pred_proba = mlr.predict_proba(X)

observed = pd.Series(le_target.inverse_transform(y_enc)).value_counts().reindex(class_names)
expected = pd.Series(y_pred_proba.sum(axis=0), index=class_names)

crosstab_gof = pd.DataFrame({
    'Observed (O)': observed.values.astype(int),
    'Expected (E)': expected.round(2).values,
}, index=class_names)
crosstab_gof.index.name = 'D1_daily_screen'

print(crosstab_gof.to_string())

chi2_stat = (((observed.values - expected.values) ** 2) / expected.values).sum()
df_chi    = len(class_names) - 1
p_chi     = 1 - chi2.cdf(chi2_stat, df_chi)

print(f"\nChi-square statistic : {chi2_stat:.4f}")
print(f"Degrees of freedom   : {df_chi}")
print(f"p-value              : {p_chi:.4f}")
print(f"Interpretation       : {'Model fits well (no significant discrepancy)' if p_chi >= 0.05 else 'Significant discrepancy between observed and expected'}")


                   Observed (O)  Expected (E)
D1_daily_screen                              
1-3 hours                   137      138.0100
3-5 hours                   125      126.1000
Less than 1 hour            112      109.6500
More than 5 hours            78       78.2400

Chi-square statistic : 0.0680
Degrees of freedom   : 3
p-value              : 0.9954
Interpretation       : Model fits well (no significant discrepancy)


# Publication-Style Results Table (Indicator Coding)
**Journal-format table — categorical predictors with reference levels, one column group per outcome class (vs. `1-3 hours`)**

In [15]:
# Reference (baseline) level chosen for every categorical predictor
ref_levels = {
    'age_class':                 '11-13 years',
    'A2_gender':                  'Male',
    'B1_residence':               'Urban area',
    'income_class':               'Middle Income (150K-300K)',
    'weight_class':               'Medium (45-60 kg)',
    'D2_weekend_screen':          'Less than 3 hours',
    'D5_parents_screen_time':     'Less than 3 hours',
    'D6_ai_daily':                'Do not use',
    'E5_bedtime':                 'Normal (9:30-10:30 PM)',
    'E6_wake_time':               'Normal (6:00-7:00 AM)',
    'F1_academic_satisfaction':   'Satisfactory',
    'F4_interest_reduced':        'Not decreased at all',
    'G1_mood_swings':             'Not at all',
    'G2_anxious_without_device':  'Not at all',
    'H1_mobile_while_eating':     'Never',
    'H2_appetite_change':         'No change',
    'BMI_class':                  'Normal',
    'D4_content_grouped':         'Other',
}

# Build indicator-coded design matrix, one indicator column per non-reference level
X_cat = df[features].copy()
indicator_frames = []
var_group_map = {}   # feature -> list of (indicator_col, level_label)

for feat in features:
    ref = ref_levels[feat]
    levels = [l for l in X_cat[feat].unique().tolist() if l != ref]
    levels.sort()
    indicators = pd.get_dummies(X_cat[feat], prefix=feat, drop_first=False)
    keep_cols = [f"{feat}_{lvl}" for lvl in levels]
    indicators = indicators[keep_cols].astype(float)
    indicator_frames.append(indicators)
    var_group_map[feat] = list(zip(keep_cols, levels))

X_indicators = pd.concat(indicator_frames, axis=1)
print("Indicator-coded design matrix shape:", X_indicators.shape)
X_indicators.head()


Indicator-coded design matrix shape: (452, 43)


,age_class_14-16 years,age_class_17-19 years,A2_gender_Female,B1_residence_Rural area,income_class_High Income (>300K),income_class_Low Income (≤150K),weight_class_Heavy (>60 kg),weight_class_Light (<45 kg),D2_weekend_screen_3-5 and More hours,D5_parents_screen_time_3-5 hours,...,H1_mobile_while_eating_Occasionally,H1_mobile_while_eating_Often,H1_mobile_while_eating_Very little time,H2_appetite_change_Loss of appetite,H2_appetite_change_Overeating,BMI_class_Overweight,BMI_class_Underweight,D4_content_grouped_Educational Content,D4_content_grouped_Entertainment & Media,D4_content_grouped_Social Media & Mixed Usage
0,1.0000,0.0000,1.0000,0.0000,1.0000,0.0000,0.0000,0.0000,1.0000,0.0000,...,0.0000,0.0000,0.0000,1.0000,0.0000,0.0000,0.0000,0.0000,0.0000,1.0000
1,0.0000,0.0000,1.0000,0.0000,0.0000,1.0000,0.0000,1.0000,1.0000,0.0000,...,0.0000,0.0000,0.0000,1.0000,0.0000,0.0000,1.0000,0.0000,0.0000,0.0000
2,0.0000,0.0000,1.0000,0.0000,0.0000,1.0000,0.0000,1.0000,0.0000,0.0000,...,0.0000,1.0000,0.0000,1.0000,0.0000,0.0000,1.0000,0.0000,1.0000,0.0000
3,1.0000,0.0000,1.0000,0.0000,1.0000,0.0000,0.0000,0.0000,0.0000,0.0000,...,0.0000,0.0000,0.0000,0.0000,1.0000,0.0000,0.0000,0.0000,0.0000,0.0000
4,1.0000,0.0000,1.0000,0.0000,1.0000,0.0000,0.0000,1.0000,1.0000,1.0000,...,0.0000,1.0000,0.0000,1.0000,0.0000,0.0000,1.0000,0.0000,0.0000,0.0000


In [16]:
# Fit MNLogit on the indicator-coded design matrix
X_indicators_sm = sm.add_constant(X_indicators.values.astype(float))
indicator_col_names = ['const'] + X_indicators.columns.tolist()

mnl_indicators = MNLogit(y_enc, X_indicators_sm)
result_indicators = mnl_indicators.fit(method='bfgs', maxiter=2000, disp=False)

eq_classes = class_names[1:]   # classes vs. reference '1-3 hours'

# Collect OR / CI / p-value for every indicator, per outcome class
indicator_stats = {}   # indicator_col -> {cls: (OR, lo, hi, p)}
for eq_idx, cls in enumerate(eq_classes):
    coefs = result_indicators.params[:, eq_idx]
    se    = result_indicators.bse[:, eq_idx]
    pvals = result_indicators.pvalues[:, eq_idx]
    for i, col in enumerate(indicator_col_names):
        if col == 'const':
            continue
        b = coefs[i]
        s = se[i]
        indicator_stats.setdefault(col, {})[cls] = (
            np.exp(b), np.exp(b - 1.96 * s), np.exp(b + 1.96 * s), pvals[i]
        )

print("Fitted indicator-coded MNLogit. Classes vs reference:", list(eq_classes))


Fitted indicator-coded MNLogit. Classes vs reference:

 ['3-5 hours', 'Less than 1 hour', 'More than 5 hours']


In [17]:
# Human-readable variable labels (edit freely to match your thesis wording)
var_labels = {
    'age_class':                 'Age group',
    'A2_gender':                  'Gender',
    'B1_residence':               'Residence',
    'income_class':               'Family income',
    'weight_class':               'Weight class',
    'D2_weekend_screen':          'Weekend screen time',
    'D5_parents_screen_time':     "Parents' screen time",
    'D6_ai_daily':                'Daily AI use',
    'E5_bedtime':                 'Bedtime',
    'E6_wake_time':               'Wake time',
    'F1_academic_satisfaction':   'Academic satisfaction',
    'F4_interest_reduced':        'Interest reduced (study)',
    'G1_mood_swings':             'Mood swings',
    'G2_anxious_without_device':  'Anxious without device',
    'H1_mobile_while_eating':     'Mobile use while eating',
    'H2_appetite_change':         'Appetite change',
    'BMI_class':                  'BMI class',
    'D4_content_grouped':         'Content type',
}

# Build the long-form publication table:
# one row per (variable, level), columns grouped per outcome class
records = []
for feat in features:
    ref = ref_levels[feat]
    label = var_labels.get(feat, feat)
    records.append({
        'Variables': f"{label} (ref: {ref})",
        'is_header': True,
    })
    for indicator_col, level in var_group_map[feat]:
        row = {'Variables': level, 'is_header': False}
        for cls in eq_classes:
            OR, lo, hi, p = indicator_stats[indicator_col][cls]
            row[f"{cls}__P"]  = p
            row[f"{cls}__OR"] = OR
            row[f"{cls}__lo"] = lo
            row[f"{cls}__hi"] = hi
        records.append(row)

pub_table = pd.DataFrame(records)
print(f"Publication table shape: {pub_table.shape}")
pub_table.head(10)


Publication table shape: (61, 14)


,Variables,is_header,3-5 hours__P,3-5 hours__OR,3-5 hours__lo,3-5 hours__hi,Less than 1 hour__P,Less than 1 hour__OR,Less than 1 hour__lo,Less than 1 hour__hi,More than 5 hours__P,More than 5 hours__OR,More than 5 hours__lo,More than 5 hours__hi
0,Age group (ref: 11-13 years),True,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,14-16 years,False,0.2505,2.0889,0.5946,7.3392,0.8969,1.0796,0.3389,3.4390,0.4133,1.9559,0.3921,9.7570
2,17-19 years,False,0.0081,7.0480,1.6604,29.9179,0.4793,0.5941,0.1404,2.5143,0.0020,14.8765,2.6841,82.4532
3,Gender (ref: Male),True,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Female,False,0.0660,0.3405,0.1079,1.0739,0.0304,4.1946,1.1452,15.3634,0.8055,0.8425,0.2153,3.2969
5,Residence (ref: Urban area),True,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,Rural area,False,0.0018,0.1791,0.0608,0.5271,0.2588,2.1933,0.5611,8.5737,0.0841,0.3230,0.0896,1.1643
7,Family income (ref: Middle Income (150K-300K)),True,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,High Income (>300K),False,0.5122,0.7036,0.2459,2.0134,0.0118,0.2273,0.0717,0.7205,0.8118,0.8540,0.2329,3.1315
9,Low Income (≤150K),False,0.0454,0.3063,0.0961,0.9761,0.0015,0.1655,0.0544,0.5031,0.5055,0.6400,0.1720,2.3808


In [18]:
# Render publication-style PDF: merged header groups (one per outcome class),
# sub-columns P-value / OR / 95% CI Lower / Upper — matching journal table layout
import os
from matplotlib.backends.backend_pdf import PdfPages
import matplotlib.pyplot as plt

os.makedirs("report_v3", exist_ok=True)
output_pdf_pub = "report_v3/mlr_publication_table_st.pdf"

ref_class = class_names[0]
n_groups = len(eq_classes)

# Column layout: Variables | (P-value, OR, Lower, Upper) per class
sub_cols_per_class = ['P-value', 'OR', 'Lower', 'Upper']
var_col_width = 0.26
group_col_width = (1 - var_col_width) / n_groups
sub_col_width = group_col_width / len(sub_cols_per_class)
col_widths = [var_col_width] + [sub_col_width] * (len(sub_cols_per_class) * n_groups)

def fmt_p(p):
    return f"{p:.2f}" if p >= 0.005 else "<.01"

def build_page_rows(page_df):
    header2 = ["Variables"] + sub_cols_per_class * n_groups
    body = []
    for _, r in page_df.iterrows():
        if r['is_header']:
            row = [r['Variables']] + [""] * (len(sub_cols_per_class) * n_groups)
        else:
            row = ["    " + r['Variables']]
            for cls in eq_classes:
                row += [fmt_p(r[f"{cls}__P"]), f"{r[f'{cls}__OR']:.3f}",
                        f"{r[f'{cls}__lo']:.3f}", f"{r[f'{cls}__hi']:.3f}"]
        body.append(row)
    return header2, body

rows_per_page = 20
pages = [pub_table.iloc[i:i+rows_per_page] for i in range(0, len(pub_table), rows_per_page)]

with PdfPages(output_pdf_pub) as pdf:
    for page_num, page_df in enumerate(pages):
        n_rows = len(page_df)
        fig_h = 1.6 + n_rows * 0.32
        fig, ax = plt.subplots(figsize=(16, fig_h))
        ax.axis('off')

        header2, body = build_page_rows(page_df)
        table_data = [header2] + body

        tbl = ax.table(cellText=table_data, colWidths=col_widths, loc='center', cellLoc='center',
                        bbox=[0, 0, 1, 0.92])
        tbl.auto_set_font_size(False)
        tbl.set_fontsize(7.5)

        n_cols = len(header2)

        # Style sub-header row (row 0): P-value/OR/Lower/Upper
        for c in range(n_cols):
            cell = tbl[0, c]
            cell.set_facecolor('#2c3e50')
            cell.set_text_props(color='white', fontweight='bold', fontsize=7)

        # Style body rows
        for r_idx, (_, r) in enumerate(page_df.iterrows(), start=1):
            for c in range(n_cols):
                cell = tbl[r_idx, c]
                cell.set_edgecolor('#cccccc')
                if r['is_header']:
                    cell.set_facecolor('#dbe5f1')
                    if c == 0:
                        cell.set_text_props(fontweight='bold', ha='left')
                else:
                    cell.set_facecolor('white')
                    if c == 0:
                        cell.set_text_props(ha='left')

        # Merged outcome-class group header row, placed just above the table (below the title)
        for g, cls in enumerate(eq_classes):
            x0 = var_col_width + g * group_col_width
            ax.text(x0 + group_col_width / 2, 0.955, f"{cls}  (vs. {ref_class})",
                    transform=ax.transAxes, ha='center', va='bottom',
                    fontsize=8.5, fontweight='bold')

        total_pages = len(pages)
        ax.set_title(
            f"Multinomial Logistic Regression — Daily Screen Time\n"
            f"Outcome: D1_daily_screen  (Reference category: '{ref_class}')   |   Page {page_num+1} of {total_pages}",
            fontsize=10, fontweight='bold', pad=42, y=1.0
        )
        pdf.savefig(fig, bbox_inches='tight')
        plt.close(fig)

print(f"Saved: {output_pdf_pub}  ({len(pages)} page(s), {len(pub_table)} rows)")


Saved: report_v3/mlr_publication_table_st.pdf  (4 page(s), 61 rows)
